# 01 - Bronze: PostgreSQL para Delta

Este notebook ingere as tabelas transacionais PostgreSQL para tabelas Delta na camada Bronze.

Variaveis de ambiente esperadas:

- `PG_HOST`, `PG_PORT`, `PG_DB`, `PG_SCHEMA`, `PG_USER`, `PG_PASSWORD`
- `PG_SSLMODE` opcional, padrao `require`
- `AIDP_CATALOG`, `BRONZE_SCHEMA`
- `BRONZE_WRITE_MODE`, padrao `overwrite`

### Proxima celula
Importa bibliotecas usadas para ler variaveis de ambiente, gerar identificadores de carga, baixar o driver JDBC e manipular colunas Spark.


In [ ]:
import os
import uuid
import urllib.request
from pyspark.sql import functions as F

### Proxima celula
Define parametros de execucao, credenciais via variaveis de ambiente, lista de tabelas de origem e nomes das tabelas Bronze de destino.


In [ ]:
CATALOG = os.getenv('AIDP_CATALOG', 'main')
BRONZE_SCHEMA = os.getenv('BRONZE_SCHEMA', 'bronze')
BRONZE_WRITE_MODE = os.getenv('BRONZE_WRITE_MODE', 'overwrite')

PG_HOST = os.environ['PG_HOST']
PG_PORT = os.getenv('PG_PORT', '5432')
PG_DB = os.environ['PG_DB']
PG_SCHEMA = os.getenv('PG_SCHEMA', 'public')
PG_USER = os.environ['PG_USER']
PG_PASSWORD = os.environ['PG_PASSWORD']
PG_SSLMODE = os.getenv('PG_SSLMODE', 'require')

SOURCE_SYSTEM = os.getenv('SOURCE_SYSTEM', 'postgresql_tpcc')
BATCH_ID = os.getenv('BATCH_ID', str(uuid.uuid4()))

SOURCE_TABLES = [
    ('warehouse', 'bronze_warehouse'),
    ('district', 'bronze_district'),
    ('customer', 'bronze_customer'),
    ('history', 'bronze_history'),
    ('item', 'bronze_item'),
    ('stock', 'bronze_stock'),
    ('orders', 'bronze_orders'),
    ('new_orders', 'bronze_new_orders'),
    ('order_line', 'bronze_order_line'),
]

### Proxima celula
Cria funcoes auxiliares para montar nomes qualificados no catalogo AIDP e nomes seguros para objetos PostgreSQL.


In [ ]:
def q_namespace(catalog, schema):
    return f'`{catalog}`.`{schema}`'


def q_table(catalog, schema, table):
    return f'`{catalog}`.`{schema}`.`{table}`'


def quote_pg_identifier(value):
    return chr(34) + value.replace(chr(34), chr(34) + chr(34)) + chr(34)


def postgres_dbtable(schema, table):
    qualified = f'{quote_pg_identifier(schema)}.{quote_pg_identifier(table)}'
    return f'(SELECT * FROM {qualified}) AS src'

### Proxima celula
Baixa, registra e distribui o driver JDBC do PostgreSQL para que o Spark consiga acessar a origem via JDBC com suporte a SSL.


In [ ]:
JDBC_JAR_PATH = os.getenv('POSTGRES_JDBC_JAR_PATH', '/tmp/postgresql-42.7.4.jar')
JDBC_JAR_URL = os.getenv(
    'POSTGRES_JDBC_JAR_URL',
    'https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.4/postgresql-42.7.4.jar',
)

try:
    spark._jvm.java.lang.Class.forName('org.postgresql.Driver')
    print('PostgreSQL JDBC driver already available on cluster classpath')
except Exception:
    if not os.path.exists(JDBC_JAR_PATH):
        urllib.request.urlretrieve(JDBC_JAR_URL, JDBC_JAR_PATH)

    gw = spark._sc._gateway
    jar_url = spark._jvm.java.io.File(JDBC_JAR_PATH).toURI().toURL()
    url_array = gw.new_array(spark._jvm.java.net.URL, 1)
    url_array[0] = jar_url
    class_loader = spark._jvm.java.net.URLClassLoader(
        url_array,
        spark._jvm.java.lang.ClassLoader.getSystemClassLoader(),
    )
    spark._jvm.java.lang.Thread.currentThread().setContextClassLoader(class_loader)
    driver_class = spark._jvm.java.lang.Class.forName('org.postgresql.Driver', True, class_loader)
    spark._jvm.java.sql.DriverManager.registerDriver(driver_class.newInstance())
    spark._jsc.addJar(JDBC_JAR_PATH)

jdbc_params = f'?sslmode={PG_SSLMODE}' if PG_SSLMODE else ''
JDBC_URL = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}{jdbc_params}'

### Proxima celula
Define funcoes reutilizaveis para ler uma tabela PostgreSQL, adicionar metadados Bronze e gravar o resultado como tabela Delta.


In [ ]:
def read_postgres_table(source_table):
    return (
        spark.read
        .format('jdbc')
        .option('url', JDBC_URL)
        .option('driver', 'org.postgresql.Driver')
        .option('user', PG_USER)
        .option('password', PG_PASSWORD)
        .option('dbtable', postgres_dbtable(PG_SCHEMA, source_table))
        .load()
    )


def add_bronze_metadata(df, source_table):
    data_columns = df.columns
    record_hash = F.sha2(
        F.concat_ws('||', *[
            F.coalesce(F.col(column_name).cast('string'), F.lit(''))
            for column_name in data_columns
        ]),
        256,
    )
    return (
        df
        .withColumn('ingestion_timestamp', F.current_timestamp())
        .withColumn('source_system', F.lit(SOURCE_SYSTEM))
        .withColumn('source_table', F.lit(source_table))
        .withColumn('batch_id', F.lit(BATCH_ID))
        .withColumn('record_hash', record_hash)
    )


def write_delta_table(df, target_table):
    (
        df.write
        .format('delta')
        .mode(BRONZE_WRITE_MODE)
        .option('overwriteSchema', 'true')
        .saveAsTable(q_table(CATALOG, BRONZE_SCHEMA, target_table))
    )

### Proxima celula
Cria o schema Bronze se necessario e executa o loop de ingestao de todas as tabelas transacionais para Delta.


In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {q_namespace(CATALOG, BRONZE_SCHEMA)}')

for source_table, target_table in SOURCE_TABLES:
    source_df = read_postgres_table(source_table)
    bronze_df = add_bronze_metadata(source_df, source_table)
    write_delta_table(bronze_df, target_table)
    print(f'Wrote {q_table(CATALOG, BRONZE_SCHEMA, target_table)} from {PG_SCHEMA}.{source_table}')

### Proxima celula
Monta um payload JSON de sucesso para uso em workflows AIDP ou para exibicao quando executado interativamente.


In [ ]:
import json

payload = {
    'status': 'SUCCESS',
    'layer': 'bronze',
    'catalog': CATALOG,
    'schema': BRONZE_SCHEMA,
    'batch_id': BATCH_ID,
    'tables': [target for _, target in SOURCE_TABLES],
}

try:
    oidlUtils.notebook.exit(json.dumps(payload))
except NameError:
    print(json.dumps(payload))